In [66]:
d_model = 64        # hidden size
history_len = 30
forecast_len = 7


GRN (Gated Residual Network)

Encounters: Inside the VSN.
What it does: Smart non-linear processing.

Action: It's a building block. It takes an input vector and:
Tries to process it with dense layers + ELU (non-linearity).
Uses a Gate (sigmoid) to decide: "Should I use this processed version, or just keep the original input?"

Why: It allows the network to learn complex patterns when needed, or just pass simple signals through untouched.

In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GRN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, output_dim)
        self.fc2 = nn.Linear(output_dim, output_dim)

        self.gate = nn.Linear(output_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)

        self.project_residual = (
            nn.Linear(input_dim, output_dim)
            if input_dim != output_dim else None
        )

    def forward(self, x):
        residual = x if self.project_residual is None else self.project_residual(x)

        x = F.elu(self.fc1(x))
        x = self.fc2(x)

        gate = torch.sigmoid(self.gate(x))
        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)

        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)


VariableSelectionNetwork (VSN)
Called Third: self.past_vsn(...) and self.future_vsn(...)

Role: The Smart Filter.

Action: It looks at the separate feature embeddings from the FeatureEncoder and asks: "Which of these features are actually useful right now?"
It uses a Weight Generator (a sub-network) to assign an importance score (e.g., Price: 0.8, Temp: 0.1, Noise: 0.1).
It creates a weighted combination of the features.

Result: A single, clean vector for each time step containing only the relevant information.
Sub-class used: GRN (Gated Residual Network)
Used by the VSN to process features and generate weights.
It allows the network to apply non-linear processing (using ELU activation) or skip processing entirely (using a Gate), giving it flexibility for both simple and complex patterns.

In [68]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()

        self.num_features = num_features

        # One GRN per feature
        self.feature_grns = nn.ModuleList([
            GRN(d_model, d_model) for _ in range(num_features)
        ])

        # Weight generator
        self.weight_grn = GRN(d_model * num_features, d_model)
        self.weight_fc = nn.Linear(d_model, num_features)

    def forward(self, x):
        """
        x: [B, T, num_features, d_model]
        """
        B, T, F, D = x.shape

        # Transform each feature
        transformed = []
        for i in range(F):
            transformed.append(self.feature_grns[i](x[:, :, i, :]))

        transformed = torch.stack(transformed, dim=2)  # [B, T, F, D]

        # Compute weights
        flat = transformed.reshape(B, T, F * D)
        weights = self.weight_grn(flat)
        weights = self.weight_fc(weights)
        weights = torch.softmax(weights, dim=-1)  # [B, T, F]

        # Weighted sum
        output = torch.sum(transformed * weights.unsqueeze(-1), dim=2)

        return output


StaticEncoder
Called First: self.static_encoder(static)

Role: Metadata Embedder.

Action: It takes your unchanging "static" data (like Store ID or Item ID) and transforms it into a feature vector (d_model).
Why: This gives the model "context." The model now knows what it is predicting for, before it even looks at the time-series history.

In [69]:
class StaticEncoder(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.encoder = nn.Linear(input_dim, d_model)

    def forward(self, x):
        return self.encoder(x)


FeatureEncoder 

What it does: Independent projecting.

Action: It takes your time-series data (e.g., 4 features like Price, Temp, Sales). It has 4 separate tiny distinct neural networks (Linear layers) inside it.
Feature 1 goes through Network 1.
Feature 2 goes through Network 2.
...

Output: [Batch, Time, 4, 64] — Each feature is now a rich vector of size 64, but they are still separate.

In [70]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.encoders = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

    def forward(self, x):
        """
        x: [B, T, F]
        returns: [B, T, F, d_model]
        """
        outs = []
        for i, enc in enumerate(self.encoders):
            outs.append(enc(x[:, :, i:i+1]))
        return torch.stack(outs, dim=2)


In [71]:
import torch.nn as nn


class TemporalAttention(nn.Module):
    """
    Multi-head self-attention across the time dimension.

    Receives the concatenated past + future VSN output of shape [B, T, D],
    lets every time step attend to every other time step, then applies a
    residual connection and LayerNorm for training stability.
    """

    def __init__(self, d_model: int, num_heads: int = 4):
        super().__init__()
        # batch_first=True so tensors are [B, T, D] throughout
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: [B, T, D]  –  output from the VSN stage
        Returns:
            [B, T, D]     –  context-enriched representation
        """
        attn_out, _ = self.attn(x, x, x)       # self-attention (Q = K = V = x)
        return self.layer_norm(x + attn_out)    # residual + norm


MinimalTFT (The Conductor)

Role: This is the main class that orchestrates the entire process.

Action: It accepts the raw data (static, past, future) and passes it stage-by-stage through the other components. It manages the flow from encoding $\to$ variable selection $\to$ temporal attention $\to$ prediction.

In [72]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 static_dim,
                 past_dim,
                 future_dim,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        # Encoders
        self.static_encoder = StaticEncoder(static_dim, d_model)
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # Variable selection
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # Attention
        self.temporal_attn = TemporalAttention(d_model, num_heads)

        # Output head
        self.output_layer = nn.Linear(d_model, 1)

        

    def forward(self, static, past, future):
        """
        static: [B, S]
        past:   [B, T_hist, O]
        future: [B, T_hist + T_fut, K]
        """

        B, T_hist, O = past.shape
        T_total = future.shape[1]

        # Encode
        static_emb = self.static_encoder(static)

        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)


        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)

        # Combine timeline
        temporal_input = torch.cat([past_sel, future_sel[:, T_hist:]], dim=1)

        # Attention
        attn_out = self.temporal_attn(temporal_input)

        # Forecast
        forecast = self.output_layer(attn_out[:, -forecast_len:])
        return forecast.squeeze(-1)


In [73]:
B = 4
static = torch.randn(B, 5)
past = torch.randn(B, 30, 1)
future = torch.randn(B, 37, 4)

model = MinimalTFT(5, 1, 4)
out = model(static, past, future)

print(out.shape)  # [B, 7]


torch.Size([4, 7])


Raw Inputs
[B, T, F]
    ->
FeatureEncoder
[B, T, F, D]
    ->
VariableSelectionNetwork
[B, T, D]
    ->
TemporalAttention
[B, T, D]
    ->
Output Head
[B, T_fut]


### Synthetic data generator to check basline


In [74]:
import torch
import numpy as np

def generate_batch(batch_size=32, history=30, forecast=7):
    t_total = history + forecast
    
    base = torch.randn(batch_size, 1) * 5
    
    time = torch.arange(t_total).float()
    season = torch.sin(2 * np.pi * time / 7)

    sales = base + season + 0.1 * torch.randn(batch_size, t_total)
    
    past = sales[:, :history].unsqueeze(-1)
    future_target = sales[:, history:]
    
    # known future feature: day-of-week sine signal
    known = season.unsqueeze(0).repeat(batch_size, 1)
    known = known.unsqueeze(-1)
    
    static = torch.randn(batch_size, 5)
    
    return static, past, known, future_target

In [75]:
model = MinimalTFT(5, 1, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch()

    optimizer.zero_grad()
    output = model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(step, loss.item())

0 4.355736255645752
50 0.96466064453125
100 0.374679833650589
150 0.21341051161289215
200 0.2701069414615631
250 0.11319655179977417
300 0.1577223837375641
350 0.30338725447654724
400 0.10942746698856354
450 0.1103789433836937


### Compare current TFT with LSTM(knows only past no known feature data)

In [76]:
import torch
import torch.nn as nn

class LSTMForecast(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, forecast_len=7):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, forecast_len)

    def forward(self, past):
        """
        past: [B, T_hist, 1]
        """
        _, (h, _) = self.lstm(past)
        h = h.squeeze(0)          # [B, hidden_dim]
        return self.fc(h)         # [B, forecast_len]

In [77]:
tft_model = MinimalTFT(5, 1, 1)
tft_optimizer = torch.optim.Adam(tft_model.parameters(), lr=1e-3)
criterion = nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch()

    tft_optimizer.zero_grad()
    output = tft_model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    tft_optimizer.step()

    if step % 100 == 0:
        print("TFT", step, loss.item())

TFT 0 3.845853567123413
TFT 100 0.428463876247406
TFT 200 0.13883087038993835
TFT 300 0.337903767824173
TFT 400 0.12447299808263779


In [78]:
lstm_model = LSTMForecast()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

for step in range(500):
    _, past, _, target = generate_batch()

    lstm_optimizer.zero_grad()
    output = lstm_model(past)
    loss = criterion(output, target)

    loss.backward()
    lstm_optimizer.step()

    if step % 100 == 0:
        print("LSTM", step, loss.item())

LSTM 0 4.078514575958252
LSTM 100 1.0153111219406128
LSTM 200 0.3462762236595154
LSTM 300 0.09614674746990204
LSTM 400 0.11139721423387527


TFT converges early, but at 400 steps they almost hav similar loss because the synthetic data is simple even LSTM can learn periodic patterns easily. So attention dont dominate here.


Upgrade the Synthetic Data

Modify generator to include:

1️⃣ Regime change

Seasonality shifts mid-series.

2️⃣ Promotion spikes (known future feature)

Add binary spike days.

3️⃣ Static-based variation

Make different static inputs change amplitude.

This is where:

Variable selection

Static context

Known future features

start to matter.

Why This Matters

Right now:

LSTM can memorize simple periodicity.

But if:

Regime changes

External known features exist

Different entities behave differently

Then:

TFT should become clearly superior.

In [79]:
import torch
import numpy as np

def generate_batch_v2(batch_size=32, history=30, forecast=7):
    T_total = history + forecast
    
    # ---------- STATIC ----------
    static = torch.randn(batch_size, 5)
    amplitude = 5 + static[:, 0:1] * 2  # [B,1]
    
    # ---------- TIME ----------
    t = torch.arange(T_total).float()
    
    weekly = torch.sin(2 * np.pi * t / 7)          # [T]
    monthly = torch.sin(2 * np.pi * t / 30)        # [T]
    
    regime = torch.where(
        t > T_total // 2,
        torch.tensor(1.5),
        torch.tensor(1.0)
    )  # [T]
    
    # ---------- PROMO ----------
    promo = torch.zeros(batch_size, T_total)
    for b in range(batch_size):
        promo_days = torch.randint(0, T_total, (3,))
        promo[b, promo_days] = 1.0
    
    # ---------- SALES ----------
    # Broadcasting works automatically
    base = amplitude * weekly  # [B, T]
    base += 0.5 * monthly      # monthly broadcasts
    
    sales = base * regime      # regime broadcasts
    sales += 8 * promo
    sales += 0.5 * torch.randn(batch_size, T_total)
    
    # ---------- SPLIT ----------
    past = sales[:, :history].unsqueeze(-1)
    target = sales[:, history:]
    
    known = torch.stack([
        weekly.unsqueeze(0).repeat(batch_size, 1),
        monthly.unsqueeze(0).repeat(batch_size, 1),
        promo
    ], dim=-1)
    
    return static, past, known, target

In [80]:
tft_model = MinimalTFT(5, 1, 3)
tft_optimizer = torch.optim.Adam(tft_model.parameters(), lr=1e-3)
criterion = nn.L1Loss()

for step in range(500):
    static, past, known, target = generate_batch_v2()

    tft_optimizer.zero_grad()
    output = tft_model(static, past, known)
    loss = criterion(output, target)

    loss.backward()
    tft_optimizer.step()

    if step % 100 == 0:
        print("TFT", step, loss.item())

TFT 0 5.000813007354736
TFT 100 0.826964795589447
TFT 200 0.6102520823478699
TFT 300 0.49807626008987427
TFT 400 0.48914676904678345


In [81]:
lstm_model = LSTMForecast()
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

for step in range(500):
    _, past, _, target = generate_batch_v2()

    lstm_optimizer.zero_grad()
    output = lstm_model(past)
    loss = criterion(output, target)

    loss.backward()
    lstm_optimizer.step()

    if step % 100 == 0:
        print("LSTM", step, loss.item())

LSTM 0 5.004086017608643
LSTM 100 1.8490962982177734
LSTM 200 1.4947763681411743
LSTM 300 1.1989067792892456
LSTM 400 0.921082079410553


Why TFT wins here

Your new synthetic includes:

✅ Static-dependent amplitude

TFT sees static → scales properly
LSTM doesn’t use static at all.

✅ Promotion spikes (known future feature)

TFT sees promo directly in future window.
LSTM must guess spikes from past.

That’s almost impossible.

✅ Regime change

Attention helps detect:

time shift

pattern boundary

LSTM struggles with longer shifts.

✅ Multi-frequency seasonality

TFT can:

weigh weekly vs monthly differently

LSTM has to encode everything in hidden state.

In [116]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 num_items,
                 num_stores,
                 num_cats,
                 num_states,
                 past_dim,
                 future_dim,
                 forecast_len=7,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        self.forecast_len = forecast_len
        self.d_model = d_model

        # -------- Static Embeddings --------
        self.item_emb = nn.Embedding(num_items, d_model)
        self.store_emb = nn.Embedding(num_stores, d_model)
        self.cat_emb = nn.Embedding(num_cats, d_model)
        self.state_emb = nn.Embedding(num_states, d_model)

        self.static_proj = nn.Linear(d_model * 4, d_model)

        # -------- Encoders --------
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # -------- Variable Selection --------
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # -------- Temporal Attention --------
        self.temporal_attn = TemporalAttention(d_model, num_heads)

        # -------- Output Head --------
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, static, past, future):
        """
        static: [B, 4]  (Long)
        past:   [B, T_hist, past_dim]
        future: [B, T_hist + T_fut, future_dim]
        """

        B, T_hist, _ = past.shape
        T_total = future.shape[1]

        # -------- Static Embedding --------
        item_vec = self.item_emb(static[:, 0])
        store_vec = self.store_emb(static[:, 1])
        cat_vec = self.cat_emb(static[:, 2])
        state_vec = self.state_emb(static[:, 3])

        static_concat = torch.cat(
            [item_vec, store_vec, cat_vec, state_vec],
            dim=-1
        )

        static_emb = self.static_proj(static_concat)  # [B, d_model]

        # -------- Encode Temporal Features --------
        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)

        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)

        # Combine past + future forecast window
        temporal_input = torch.cat(
            [past_sel, future_sel[:, T_hist:]],
            dim=1
        )

        # -------- Inject Static Context --------
        static_expanded = static_emb.unsqueeze(1).expand(
            -1,
            temporal_input.size(1),
            -1
        )

        temporal_input = temporal_input + static_expanded

        # -------- Attention --------
        attn_out = self.temporal_attn(temporal_input)

        # -------- Forecast --------
        forecast = self.output_layer(
            attn_out[:, -self.forecast_len:]
        )

        return forecast.squeeze(-1)

In [117]:
import pandas as pd
import numpy as np

sales = pd.read_csv("sales_train_validation.csv")
calendar = pd.read_csv("calendar.csv")

In [118]:
num_items = sales["item_id"].nunique()
num_stores = sales["store_id"].nunique()
num_cats = sales["cat_id"].nunique()
num_states = sales["state_id"].nunique()
num_items,num_stores,num_cats,num_states

(3049, 10, 3, 3)

In [119]:
# Select first 200 series only
sales = sales.iloc[:200]

We select:
- A subset of series (e.g., first 200).
- The last 400 days of sales data.

This keeps the dataset computationally manageable while preserving recent temporal patterns.

In [120]:
sales_values = sales.iloc[:, -400:].values   # shape: [200, 400]

In [121]:
calendar_subset = calendar.iloc[-400:]

In [122]:
known_features = calendar_subset[[
    "wday",
    "month",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]].values

From the calendar dataset, we extract known future features such as:

- `wday` (weekday index)
- `month`
- `snap_CA`, `snap_TX`, `snap_WI`

These features are known ahead of time and will be provided to the model for both the historical and forecast window.

In [123]:
means = sales_values.mean(axis=1, keepdims=True)
stds = sales_values.std(axis=1, keepdims=True) + 1e-6

sales_norm = (sales_values - means) / stds

Retail demand varies significantly across products.

We normalize each time series independently:

\[
x_{norm} = \frac{x - \mu}{\sigma}
\]

This stabilizes training and prevents high-volume items from dominating gradients.

In [124]:
for col in ["item_id", "store_id", "cat_id", "state_id"]:
    sales[col] = sales[col].astype("category").cat.codes

In [125]:
static_features = sales[["item_id", "store_id", "cat_id", "state_id"]].values
static_tensor = torch.tensor(static_features).long().to(device)

In [126]:
history = 30
forecast = 7
T_total = sales_norm.shape[1]

train_cutoff = 350  # first 350 days train
val_start = 350     # validation starts here

train_samples = []
val_samples = []

for i in range(sales_norm.shape[0]):  # for each series
    for t in range(history, T_total - forecast):

        past = sales_norm[i, t-history:t]
        target = sales_norm[i, t:t+forecast]
        known = known_features[t-history:t+forecast]

        if t < train_cutoff:
            train_samples.append((i, past, known, target))
        else:
            val_samples.append((i, past, known, target))

In [127]:
def build_tensors(samples):
    series_ids = torch.tensor([s[0] for s in samples])
    past_tensor = torch.tensor([s[1] for s in samples]).float().unsqueeze(-1)
    known_tensor = torch.tensor([s[2] for s in samples]).float()
    target_tensor = torch.tensor([s[3] for s in samples]).float()

    return series_ids, past_tensor, known_tensor, target_tensor

train_ids, train_past, train_known, train_target = build_tensors(train_samples)
val_ids, val_past, val_known, val_target = build_tensors(val_samples)

We convert the sliding window samples into PyTorch tensors:

- `past_tensor` → [N, 30, 1]
- `known_tensor` → [N, 37, K]
- `target_tensor` → [N, 7]
- `series_ids` → used to fetch static features

This prepares the dataset for model training.

In [128]:
train_ids = train_ids.to(device)
train_past = train_past.to(device)
train_known = train_known.to(device)
train_target = train_target.to(device)

val_ids = val_ids.to(device)
val_past = val_past.to(device)
val_known = val_known.to(device)
val_target = val_target.to(device)

In [133]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.L1Loss()

batch_size = 64
model = MinimalTFT(
    num_items=num_items,
    num_stores=num_stores,
    num_cats=num_cats,
    num_states=num_states,
    past_dim=1,
    future_dim=5,
    forecast_len=7,
    d_model=64
).to(device)

for epoch in range(10):

    model.train()
    perm = torch.randperm(len(train_past))

    for i in range(0, len(train_past), batch_size):
        idx = perm[i:i+batch_size]

        static = static_tensor[train_ids[idx]]
        past = train_past[idx]
        known = train_known[idx]
        target = train_target[idx]

        optimizer.zero_grad()
        output = model(static, past, known)
        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

    # ---- VALIDATION ----
    model.eval()
    with torch.no_grad():
        static_val = static_tensor[val_ids]
        val_output = model(static_val, val_past, val_known)
        val_loss = criterion(val_output, val_target)

    print(f"Epoch {epoch} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    print(val_output.mean().item())

Epoch 0 | Train Loss: 1.2106 | Val Loss: 1.1901
0.8437376618385315
Epoch 1 | Train Loss: 1.2542 | Val Loss: 1.1901
0.8437376618385315
Epoch 2 | Train Loss: 1.2349 | Val Loss: 1.1901
0.8437376618385315
Epoch 3 | Train Loss: 1.2099 | Val Loss: 1.1901
0.8437376618385315
Epoch 4 | Train Loss: 1.1905 | Val Loss: 1.1901
0.8437376618385315
Epoch 5 | Train Loss: 1.1722 | Val Loss: 1.1901
0.8437376618385315
Epoch 6 | Train Loss: 1.2256 | Val Loss: 1.1901
0.8437376618385315
Epoch 7 | Train Loss: 1.1832 | Val Loss: 1.1901
0.8437376618385315
Epoch 8 | Train Loss: 1.1986 | Val Loss: 1.1901
0.8437376618385315
Epoch 9 | Train Loss: 1.2063 | Val Loss: 1.1901
0.8437376618385315


In [134]:
print(static_tensor.shape)
print(val_ids.shape)
print(val_ids.max(), val_ids.min())

torch.Size([200, 4])
torch.Size([8600])
tensor(199, device='cuda:0') tensor(0, device='cuda:0')
